In [1]:
import json
import pandas as pd
import requests
from dotenv import load_dotenv
import os

In [2]:
def get_fmp_pricing_data(ticker: str, start_date: str, end_date: str, fmp_api_key: str):
    """
    Fetches historical pricing data for a given ticker from the Financial Modeling Prep (FMP) API.

    Parameters:
    ---------
    ticker : str
        The stock or asset ticker symbol (e.g., "AAPL").
    start_date : str
        The start date for retrieving historical data (format: "YYYY-MM-DD").
    end_date : str
        The end date for retrieving historical data (format: "YYYY-MM-DD").
    fmp_api_key : str
        The API key for accessing the Financial Modeling Prep API.

    Returns:
    -------
    pd.DataFrame
        DataFrame with historical prices indexed by date.
    """

    url = f'https://financialmodelingprep.com/stable/historical-price-eod/full?symbol={ticker}&from={start_date}&to={end_date}&apikey={fmp_api_key}'
    response = requests.get(url)
    response.raise_for_status()
    data = response.json()
    if not isinstance(data, list) or len(data) == 0:
        raise ValueError(f'Unexpected API response: {data}')
    prices = pd.DataFrame(data)
    prices['date'] = pd.to_datetime(prices['date'])
    prices = prices[(prices['date'] >= start_date) & (prices['date'] <= end_date)]
    prices = prices.set_index('date').sort_index()
    #for col in ['price', 'volume']:
    #    if col in prices.columns:
    #        prices[col] = pd.to_numeric(prices[col], errors='coerce')
    return prices

In [3]:
_ = load_dotenv()  
df = get_fmp_pricing_data(
    ticker='SPY', 
    start_date='2025-01-01', 
    end_date='2025-06-01', 
    fmp_api_key=os.environ['FSM_API_KEY']
)

In [4]:
df.head(3)

,symbol,open,high,low,close,volume,change,changePercent,vwap
date,,,,,,,,,
2025-01-02,SPY,589.39,591.13,580.50,584.64,50204000,-4.75,-0.80592,586.42
2025-01-03,SPY,587.53,592.60,586.43,591.95,37888500,4.42,0.75230,589.63
2025-01-06,SPY,596.27,599.70,593.60,595.36,47679400,-0.91,-0.15262,596.23
